# 07 — The confusion matrix, precision & recall

*Module 00 · Getting Started — notebook 7 of 11*

Accuracy told us *how many* mistakes a model makes. This notebook opens up *which* mistakes — and turns that into the two numbers practitioners reach for most: precision and recall.

**Prerequisites:** 06 (accuracy, baseline, the positive class).

**What you'll be able to do**
- Read a **confusion matrix** and name its four cells (TP, FP, FN, TN).
- Compute **precision** and **recall** by hand, and with scikit-learn.
- Combine them with **F1**, and say what each one measures.
- Decide, for a given problem, whether precision or recall matters more.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestCentroid

from ml_course import datasets, viz

np.random.seed(0)
viz.use_course_style()

X, y = datasets.penguins_xy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y
)

POSITIVE = 'Gentoo'
LABELS = ['Adelie', 'Gentoo']  # order: negative, positive

# A deliberately weaker model: nearest centroid on BILL LENGTH ALONE, so it makes errors to study.
bill = ['bill_length_mm']
clf = NearestCentroid().fit(X_train[bill], y_train)
pred = clf.predict(X_test[bill])

## Where we are

Notebook 06 left us with a complaint: accuracy collapses every mistake into one number and cannot tell a missed Gentoo from a false alarm. To fix that we need a model that actually *makes* some mistakes to look at — and our two-feature classifier was perfect on the test set. So we deliberately weaken it.

## Why bill length alone

Notebook 03 showed that flipper length separates the two species cleanly, while **bill length overlaps** more. If we classify using bill length *only*, the nearest-centroid rule gets most penguins right but trips on the ones in the overlap — exactly the handful of errors we need to study. This is a teaching device: a deliberately weaker model, not the one we would ship.

In [ ]:
print(f'accuracy (bill length only): {accuracy_score(y_test, pred):.3f}')
print('for comparison, the two-feature model from notebooks 05-06 scored 1.000')

### Read the output

Using bill length alone, accuracy slips to **0.957** — a little below the two-feature 1.00, and that is the point: there are now a few errors. The question this notebook answers is what *kind* of errors they are.

## The confusion matrix

The **confusion matrix** lays predictions against the truth in a small table: each row is the true class, each column the predicted class. With two classes and **Gentoo as the positive class** (our choice from notebook 06), the four cells have names:

- **True positive (TP)** — a Gentoo correctly called Gentoo.
- **False positive (FP)** — an Adélie wrongly called Gentoo (a *false alarm*).
- **False negative (FN)** — a Gentoo wrongly called Adélie (a *miss*).
- **True negative (TN)** — an Adélie correctly called Adélie.

In [ ]:
cm = confusion_matrix(y_test, pred, labels=LABELS)
viz.plot_confusion_matrix(cm, LABELS)
plt.show()

tn, fp, fn, tp = cm[0, 0], cm[0, 1], cm[1, 0], cm[1, 1]
print(f'true negatives  (Adelie called Adelie): {tn}')
print(f'false positives (Adelie called Gentoo): {fp}')
print(f'false negatives (Gentoo called Adelie): {fn}')
print(f'true positives  (Gentoo called Gentoo): {tp}')

### Read the figure

Our model's errors now have names. Reading the matrix: **37** of the 38 test Adélies correctly identified (TN) and **29** of the 31 Gentoos caught (TP), against only two kinds of mistake — **1** Adélie raising a false alarm (FP) and **2** Gentoos slipping through (FN). Three errors out of 69, but of *two different kinds*. Accuracy added them together; the matrix keeps them apart, and that is what lets us build sharper measures.

## Precision

**Precision** asks: of all the penguins we *called* Gentoo, how many really are?

$$ \text{precision} = \frac{TP}{TP + FP}. $$

It looks only at the positive *predictions*, and it falls when the model raises false alarms (FP).

In [ ]:
precision_by_hand = tp / (tp + fp)
print(f'precision by hand : {precision_by_hand:.3f}   = TP / (TP + FP) = {tp} / {tp + fp}')
print(f'precision_score   : {precision_score(y_test, pred, pos_label=POSITIVE):.3f}')

### Read the output

Precision is **0.967**: of the 30 penguins this model labelled Gentoo, 29 truly were. When it says "Gentoo", trust it about 97% of the time. If acting on a positive prediction were costly — a treatment, an arrest, a shutdown — precision is the number you would watch.

## Recall

**Recall** asks the complementary question: of all the *actual* Gentoos, how many did we catch?

$$ \text{recall} = \frac{TP}{TP + FN}. $$

It looks only at the true positives in the data, and it falls when the model misses them (FN).

In [ ]:
recall_by_hand = tp / (tp + fn)
print(f'recall by hand : {recall_by_hand:.3f}   = TP / (TP + FN) = {tp} / {tp + fn}')
print(f'recall_score   : {recall_score(y_test, pred, pos_label=POSITIVE):.3f}')

### Read the output

Recall is **0.935**: we caught 29 of the 31 real Gentoos, and missed two. Precision and recall are not the same number, and they answer different questions — "are our positive calls trustworthy?" versus "did we find all the positives?" Improving one often costs the other, a tension we make concrete with thresholds in notebook 08.

## F1: one number for both

Sometimes you want a single score that rewards being good at *both*. **F1** is the harmonic mean of precision and recall:

$$ F_1 = 2 \cdot \frac{\text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}. $$

The harmonic mean sits near the *smaller* of the two, so F1 is high only when precision and recall are *both* high — you cannot fake it by acing one and failing the other.

In [ ]:
f1_by_hand = 2 * precision_by_hand * recall_by_hand / (precision_by_hand + recall_by_hand)
print(f'F1 by hand : {f1_by_hand:.3f}')
print(f'f1_score   : {f1_score(y_test, pred, pos_label=POSITIVE):.3f}')

### Read the output, and which to optimize

F1 is **0.951** — high, because precision and recall are both high here.

Which of the three should you optimize? It depends on what an error costs. A spam filter leans on **precision**: a false positive bins a real email, far worse than letting one spam through. Cancer screening leans on **recall**: a false negative misses a sick patient, far worse than a false alarm that a follow-up test clears. F1 is a sensible default when the two errors cost about the same. The metric you choose is a statement about consequences, not a technicality.

## Your turn

1. A spam classifier (positive = "spam") produces this confusion matrix: TP = 80, FP = 10, FN = 20, TN = 390. Compute its precision and its recall.
2. Name one problem where **recall** matters more than precision, and one where **precision** matters more. Justify each in a line.
3. If we had chosen **Adélie** as the positive class instead of Gentoo, which of TP / FP / FN / TN would swap with which? (You don't need to recompute — only describe.)

Notebook 08 shows how moving a single dial trades precision for recall.

## What you built

You can now look past a single accuracy figure to the *structure* of a model's mistakes:

- You read a **confusion matrix** and named its four cells.
- You computed **precision** (are our positive calls right?) and **recall** (did we catch the positives?), by hand and with scikit-learn.
- You combined them with **F1**, and you can argue which to favour for a given problem.

**Vocabulary added:** confusion matrix, true/false positive, true/false negative, precision, recall, F1, asymmetric error costs.

## References

- A. Géron, *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd ed., O'Reilly, 2022 — chapter 3 (the confusion matrix, precision and recall, the precision/recall trade-off, F1).
- scikit-learn developers, `sklearn.metrics` — `confusion_matrix`, `precision_score`, `recall_score`, `f1_score`.

---
Previous: **06 — Is it any good? accuracy and a baseline** · Next: **08 — Scores, thresholds, ROC & AUC**